In [ ]:
import sys
sys.path.append('/work/SWOT/swotdev/swotdev/stephag/floodplain_dem/scripts')
sys.path.append('/work/SWOT/swotdev/swotdev/stephag/floodplain_dem/src')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import skimage
from skimage.morphology import erosion, dilation

In [ ]:
import mrf_waterland_toolbox as toolbox
from mrf_method import (get_grid_lat_lon_ref, extract_params_from_pixc, 
                        plot_var_and_zoom, apply_mrf_method, get_labels, get_mean_dem,
                        write_mrf_fpdem_file)

In [ ]:
path_pixc_list = ["/work/EXPERT_CENTER/mwec/workspace/HR/science_orbit_sites_public/Bijagos/pixc"]
#output_file = "/work/EXPERT_CENTER/mwec/workspace/HR/science_orbit_sites_public/Bijagos/fpdem_mean_test.nc"
output_file = "/work/SWOT/swotdev/swotdev/stephag/floodplain_dem/testcases/Bijagos/fpdem_mean_test.nc"

In [ ]:
# Choose to use a referential DEM file to get the grid or
# 
refdem_grid = False
# Optional if refdem_grid = False
refdem_file = "/work/EXPERT_CENTER/mwec/data/aux_files/sad/production/RefDEM/SWOT_RefDEM_20000101T000000_21000101T000000_20220301T025755_v101/Nom/197/SWOT_RefDEM_Nom_197_174L_20000101T000000_21000101T000000_20220301T025755_v101.nc"
# Optional if refdem_grid = True
step = 50. # Choose to define the grid based on the step size defined here (in meters)
margin = [0., 0.]

# Get all PIXC path and filename
for path in path_pixc_list:
    for root, dir, files in os.walk(path):
        for file in files:
            if file.startswith("SWOT_L2_HR_PIXC_") & file.endswith(".nc"):
                pixc = os.path.join(root, file)

ref_dem, latitude_ref_dem, longitude_ref_dem = get_grid_lat_lon_ref(refdem_grid, 
                                                                    path_ref_dem=refdem_file, 
                                                                    pixc=pixc, step=step, margin=margin)
print(ref_dem)

In [ ]:
plot = True
# Select a small part of the region (range/azimuth indexes) for visualization
l0, l1 = 150, 400
c0, c1 = 300, 700
zoom = (slice(l0, l1), slice(c0, c1))

fpdem = []
for path in path_pixc_list:
    for root, dir, files in os.walk(path):
        for file in files:
            if file.startswith("SWOT_L2_HR_PIXC_") & file.endswith(".nc"):
                print(path, file)
                
                (grid_h, grid_h_interp, 
                 grid_inc, 
                 grid_sig0, grid_sig0_interp, 
                 grid_coh_interp, 
                 mask, 
                 date, 
                 grid_coh_th_interp) = extract_params_from_pixc(file, root, ref_dem)

                if plot:
                    fig, axes = plt.subplots(1, 2, figsize=(10, 5)) # 1 ligne, 2 colonnes
                    im1 = axes[0].imshow(10*np.log10(grid_sig0)[zoom], cmap="Greys_r")
                    im2 = axes[1].imshow(grid_h[zoom], vmin=25,vmax=30)

                fpdem.append([grid_h, grid_h_interp, grid_inc, grid_sig0, grid_sig0_interp, 
                              grid_coh_interp, mask, date, grid_coh_th_interp])

In [ ]:
# Sorted by date
fpdem = sorted(fpdem, key=lambda fpdem: fpdem[7])

In [ ]:
# Plot sig0 before and after erosion/dilation for the first date, on the whole region
tab_tmp = np.where(fpdem[4][0] == 0., 0, 1)
kernel = np.ones([5,5])
tab_tmp2 = erosion(dilation(tab_tmp, kernel), kernel)

fig, axes = plt.subplots(1, 2, figsize=(10, 5)) # 1 ligne, 2 colonnes
axes[0].imshow(tab_tmp)
axes[1].imshow(tab_tmp2)

In [ ]:
# Unit test on one date

# ------------------------------------------
# 4. PROCESSING (MRF)
# ------------------------------------------

pixel_res_m = 30.0
tile_size_km = 25
stride_km = 20.0
border_exclude_km = 2.5

land_law = 'gaussian'  # Options: 'gaussian' or 'exponnorm'
weight_ssh = 0.5
max_iters = 50
convergence = 0.02

p_sig0_init = {
    'mu_land': 7.0, 'std_land': 3.5,
    'mu_water': 10.0, 'std_water': 2.5
}

p_ssh_init = {
    'mu_land': 0.25, 'std_land': 1.,
    'mu_water': 0.0, 'std_water': 0.1,
    'expon_k': 1.0, 'expon_loc': 0.1, 'expon_scale': 0.05
}

print('Date: ', fpdem[4][7])

(height_cycle_0, sig0_cycle_0,
 proba_smooth_0, labels_smooth_0, 
 proba_smooth_h, labels_smooth_h) = apply_mrf_method(
    fpdem[4],
    p_sig0_init, p_ssh_init, land_law, max_iters, convergence, weight_ssh,
    pixel_res_m, tile_size_km, stride_km, border_exclude_km
)

tab_tmp_orig = np.where(fpdem[4][0]==0., 0, 1)
kernel_dilate = np.ones([3,3])
kernel_erosion = np.ones([3,3])
tab_tmp = erosion(dilation(tab_tmp_orig, kernel_dilate), kernel_erosion)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))  # 1 ligne, 2 colonnes
axes[0].imshow(tab_tmp_orig)
axes[1].imshow(tab_tmp2)


In [ ]:
# Visualization of the unit test, on the whole region and zoomed part
if plot:
    
    # sig0
    plot_var_and_zoom(sig0_cycle_0, "sig0_cycle_0", cmap="Greys_r", zoom=zoom)
     
    # Height
    plot_var_and_zoom(height_cycle_0, "h_cycle_0", vmin=-2, vmax=2, zoom=zoom)

    # Proba_smooth
    plot_var_and_zoom(np.where(proba_smooth_h > 0, proba_smooth_h, 1), "proba_smooth_h", zoom=zoom)
    
    plot_var_and_zoom(np.where(proba_smooth_0 > 0, proba_smooth_0, 1), "proba_smooth_0", zoom=zoom)

    # Label_smooth
    plot_var_and_zoom(np.where(labels_smooth_h > -100, labels_smooth_h, -1), "labels_smooth_h", zoom=zoom)
    
    plot_var_and_zoom(np.where(labels_smooth_0 > -100, labels_smooth_0, -1), "labels_smooth_0", zoom=zoom)

    #
    fig, axes = plt.subplots(1, 3, figsize=(30, 10))  # 1 ligne, 2 colonnes
    
    axes[0].set_title("swot sig0 (dB) " + fpdem[4][7])
    axes[1].set_title("proba map without height " + fpdem[4][7])
    axes[2].set_title("proba map with height " + fpdem[4][7])

    cax0 = make_axes_locatable(axes[0]).append_axes("right", size="5%", pad=0.05)
    cax1 = make_axes_locatable(axes[1]).append_axes("right", size="5%", pad=0.05)
    cax2 = make_axes_locatable(axes[2]).append_axes("right", size="5%", pad=0.05)
    
    sig0_plot = axes[0].imshow(10*np.log10(fpdem[4][3])[zoom], vmin=0, vmax=15, cmap="Greys_r")
    labels_plot = axes[1].imshow(proba_smooth_0[zoom])
    labels_plot_with_height = axes[2].imshow(proba_smooth_h[zoom])
 
    fig.colorbar(sig0_plot, ax=axes[0], cax=cax0)
    fig.colorbar(labels_plot, ax=axes[1], cax=cax1)
    fig.colorbar(labels_plot_with_height, ax=axes[2], cax=cax2)

In [ ]:
# ------------------------------------------
# 4. PROCESSING (MRF)
# ------------------------------------------
pixel_res_m = 30.0
tile_size_km = 25
stride_km = 10.0
border_exclude_km = 2.5

land_law = 'gaussian'   # Options: 'gaussian' or 'exponnorm'
weight_ssh = 1.5
max_iters = 50
convergence = 0.02

proba_map_list = []
for i in range(len(fpdem)):
    print(f"\nApplying method on PIXC {i+1}/{len(fpdem)}")
    
    P_SIG0_INIT = {
        'mu_land': 7.0, 'std_land': 3.5,
        'mu_water': 10.0, 'std_water': 2.5
    }
    P_SSH_INIT = {
        'mu_land': 0.25, 'std_land': 1.,
        'mu_water': 0.0, 'std_water': 0.1,
        'expon_k': 1.0, 'expon_loc': 0.1, 'expon_scale': 0.05
    }

    (height_cycle_0, sig0_cycle_0,
     proba_smooth_0, labels_smooth_0, 
     proba_smooth_h, labels_smooth_h) = apply_mrf_method(
        fpdem[i],
        p_sig0_init, p_ssh_init, land_law, max_iters, convergence, weight_ssh,
        pixel_res_m, tile_size_km, stride_km, border_exclude_km
    )

    proba_map_list.append([proba_smooth_0, labels_smooth_0, proba_smooth_h, labels_smooth_h])


In [ ]:
# Plot height, sig0 and proba_smooth without and with height
for i in range(len(fpdem)):

    fig, axes = plt.subplots(1, 4, figsize=(30, 10))  # 1 row, 2 columns
    
    axes[0].set_title("swot dem " + fpdem[i][7])
    axes[1].set_title("swot sig0 (dB) " + fpdem[i][7])
    axes[2].set_title("proba map without height " + fpdem[i][7])
    axes[3].set_title("proba map with height " + fpdem[i][7])
    
    cax0 = make_axes_locatable(axes[0]).append_axes("right", size="5%", pad=0.05)
    cax1 = make_axes_locatable(axes[1]).append_axes("right", size="5%", pad=0.05)
    cax2 = make_axes_locatable(axes[2]).append_axes("right", size="5%", pad=0.05)
    cax3 = make_axes_locatable(axes[3]).append_axes("right", size="5%", pad=0.05)
    
    #plt.imshow((fpdem[i][1].data-0*ref_dem.elevation.data), vmin=250, vmax=320)
    dem_plot = axes[0].imshow((fpdem[i][0].data)[zoom], vmin=25, vmax=32)
    sig0_plot = axes[1].imshow(10*np.log10(fpdem[i][3])[zoom], vmin=0, vmax=15, cmap="Greys_r")
    labels_plot = axes[2].imshow(proba_map_list[i][0][zoom])
    labels_plot_with_height = axes[3].imshow(proba_map_list[i][2][zoom])
    
    fig.colorbar(dem_plot, ax=axes[0], cax=cax0)
    fig.colorbar(sig0_plot, ax=axes[1], cax=cax1)    
    fig.colorbar(labels_plot, ax=axes[2], cax=cax2)
    fig.colorbar(labels_plot_with_height, ax=axes[3], cax=cax3)
    

In [ ]:
# Get the combined labels and show the results on the zoomed region
threshold_without_height = 0.2
threshold_with_height = 0.2

labels_combined_list = []
for i in range(len(fpdem)):

    labels_combined = get_labels(fpdem, i, proba_map_list, 
                                 threshold_with_height, threshold_without_height, zoom)
    labels_combined_list.append(labels_combined)


In [ ]:
# Select bad dates to filter them out later
bad_dates = [20250727, 20251220, 20251018, 20251129]


In [ ]:
# Compute the mean elevation, filtering out bad dates
mean_dem, mean_sig0, mean_coh, mean_coh_th, count = get_mean_dem(fpdem.copy(), 
                                                                 labels_combined_list.copy(), 
                                                                 bad_dates)


In [ ]:
# Plot count
plot_var_and_zoom(count, "Count", zoom=zoom)


In [ ]:
# Plot mean_dem, and mean_sig0
plot_var_and_zoom(mean_dem, "Mean dem", vmin=25, vmax=30, zoom=zoom)

plot_var_and_zoom(10*np.log10(mean_sig0), "Mean sig0", cmap="Greys_r", vmin=0, vmax=20, zoom=zoom)


In [ ]:
# Plot mean sig0
fig, axes = plt.subplots(1, 1, figsize=(15, 15))
axes.set_title("Mean sig0 (log)")
cax0 = make_axes_locatable(axes).append_axes("right", size="5%", pad=0.05)
im0 = axes.imshow(10*np.log10(np.where(mean_sig0 <= 0., 0.00001, mean_sig0)),
                  cmap="Greys_r", 
                  vmin=-5, vmax=30)
fig.colorbar(im0, ax=axes, cax=cax0)

In [ ]:
write = True
epsg = "4326"
if write:
    
    write_mrf_fpdem_file(output_file, longitude_ref_dem, latitude_ref_dem, mean_dem, epsg)
    